In [1]:
!pip install -q convokit scikit-learn pandas numpy scipy pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/Break-Through-Tech/Market-Research-1A-discussion-intelligence-toolkit.git

Cloning into 'Market-Research-1A-discussion-intelligence-toolkit'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 175 (delta 69), reused 92 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 851.82 KiB | 3.11 MiB/s, done.
Resolving deltas: 100% (69/69), done.


In [3]:
import os
import sys

project_path = "/content/Market-Research-1A-discussion-intelligence-toolkit"

os.chdir(project_path)
sys.path.insert(0, os.path.join(project_path, "artifacts"))

print("Project folder connected successfully!")

Project folder connected successfully!


In [4]:
from corpora import CoarseDiscourseLoader

loader = CoarseDiscourseLoader()
splits = loader.load()

print("Dataset loaded successfully!")
print(splits.summary())
print("Categories:", splits.labels)

No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem
Dataset loaded successfully!
train=79767 dev=9825 test=9641
Categories: ['agreement', 'announcement', 'answer', 'appreciation', 'disagreement', 'elaboration', 'humor', 'negativereaction', 'question']


In [5]:
from collections import Counter
from evaluation import evaluate_classifier

# just finding which label shows up the most
train_labels = [ex.label for ex in splits.train]
most_common = Counter(train_labels).most_common(1)[0][0]

# this model just guesses the same label everytime
class MajorityBaseline:
    def predict(self, texts):
        return [most_common] * len(texts)

model = MajorityBaseline()

# checking how good the model did
results = evaluate_classifier(
    model,
    texts=[ex.text for ex in splits.test],
    labels=[ex.label for ex in splits.test]
)

print("Most common label:", most_common)
print("Accuracy:", results.accuracy)
print("Macro F1:", results.macro_f1)
print("Weighted F1:", results.weighted_f1)

Most common label: answer
Accuracy: 0.4123016284617778
Macro F1: 0.06487465726596162
Weighted F1: 0.24073134152990114


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# getting the comments and labels from the training data
X_train = [ex.text for ex in splits.train]
y_train = [ex.label for ex in splits.train]

# using tfidf and logistic regression for the next model
tfidf_model = make_pipeline(
    TfidfVectorizer(max_features=10000),
    LogisticRegression(max_iter=1000)
)

# training the model on the reddit comments
tfidf_model.fit(X_train, y_train)

# testing how good it does on the dev data first
dev_results = evaluate_classifier(
    tfidf_model,
    texts=[ex.text for ex in splits.dev],
    labels=[ex.label for ex in splits.dev]
)

print("TF-IDF + Logistic Regression Results")
print("Accuracy:", dev_results.accuracy)
print("Macro F1:", dev_results.macro_f1)
print("Weighted F1:", dev_results.weighted_f1)

TF-IDF + Logistic Regression Results
Accuracy: 0.5239694656488549
Macro F1: 0.30301685950463614
Weighted F1: 0.4808696904206123


In [7]:
# checking both models on the same dev data
baseline_dev = evaluate_classifier(
    model,
    texts=[ex.text for ex in splits.dev],
    labels=[ex.label for ex in splits.dev]
)

# comparing the scores
print("Majority Baseline vs TF-IDF")
print("Accuracy:", round(baseline_dev.accuracy, 3),
      "vs", round(dev_results.accuracy, 3))
print("Macro F1:", round(baseline_dev.macro_f1, 3),
      "vs", round(dev_results.macro_f1, 3))
print("Weighted F1:", round(baseline_dev.weighted_f1, 3),
      "vs", round(dev_results.weighted_f1, 3))

Majority Baseline vs TF-IDF
Accuracy: 0.411 vs 0.524
Macro F1: 0.065 vs 0.303
Weighted F1: 0.24 vs 0.481


In [8]:
import json

# saving the scores so we have them for later
results_data = {
    "majority_baseline": baseline_dev.model_dump(),
    "tfidf_logistic_regression": dev_results.model_dump()
}

# putting the results into a json file
with open("baseline_results.json", "w") as f:
    json.dump(results_data, f, indent=2)

print("Results saved!")

Results saved!


## Baseline Model Results

For this part of the project, I worked on two baseline models using the Coarse Discourse dataset. The first one was a majority baseline that just predicts the most common label, which was "answer." After that, I used TF-IDF and Logistic Regression to make another model that actually looks at the words in the Reddit comments.

I tested both models on the same dev dataset to compare how they did.

### Results

| Metric | Majority Baseline | TF-IDF + Logistic Regression |
|---|---|---|
| Accuracy | 0.411 | 0.524 |
| Macro F1 | 0.065 | 0.303 |
| Weighted F1 | 0.240 | 0.481 |

From the results, the TF-IDF model did better in all three areas. The accuracy went from around 41% to 52%, and the F1 scores also improved. This makes sense since the second model is learning from the actual text instead of just predicting the same label every time.

I used the training data to set up both models and the dev data to compare the results. This gives us a starting point to compare with other models later on.